# Deepfake Detection — Weights Quantization Impact
**Run on Google Colab with T4 GPU**
**Wait for img2img upload to Drive to finish before running Cell 6+**

| Cell | Purpose |
|------|----------|
| 1 | Clone repo + install |
| 2 | Mount Drive |
| 3 | Top up selected_images to 280 from FORLAB (no duplicates) |
| 4 | Copy pretrained weights |
| 5 | Patch detectors |
| 6 | Build unified dataset structure |
| 7 | Verify dataset |
| 8 | Patch launcher + run detectors |
| 9 | Collect results |
| 10 | Baseline analysis (quantization as primary axis) |
| 11 | Improvements A-E |

In [1]:
!git clone https://github.com/truebees-ai/Image-Deepfake-Detectors-Public-Library.git
%cd /content/Image-Deepfake-Detectors-Public-Library
!pip install -r requirements.txt -q

Cloning into 'Image-Deepfake-Detectors-Public-Library'...
remote: Enumerating objects: 324, done.
remote: Counting objects: 100% (103/103), done.
remote: Compressing objects: 100% (68/68), done.
remote: Total 324 (delta 51), reused 35 (delta 35), pack-reused 221 (from 1)
Receiving objects: 100% (324/324), 23.08 MiB | 13.54 MiB/s, done.
Resolving deltas: 100% (115/115), done.
Filtering content: 100% (4/4), 184.93 MiB | 18.32 MiB/s, done.
/content/Image-Deepfake-Detectors-Public-Library
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 7.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.2/797.2 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 105.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 MB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive'
print('Drive mounted')

Mounted at /content/drive
Drive mounted


In [4]:
import glob

DRIVE = '/content/drive/MyDrive'

real_files = (
    glob.glob(f'{DRIVE}/FORLAB/*.jpg') +
    glob.glob(f'{DRIVE}/FORLAB/*.png')
)
print(f'FORLAB real images: {len(real_files)}')
print(f'Sample: {real_files[0] if real_files else "empty"}')

FORLAB real images: 1702
Sample: /content/drive/MyDrive/FORLAB/00773.jpg


In [5]:
import os, shutil

WEIGHTS_DIR = f'{DRIVE}/weights'
REPO_ROOT   = '/content/Image-Deepfake-Detectors-Public-Library'

for det in ['CLIP-D', 'NPR', 'R50_nodown']:
    src     = f'{WEIGHTS_DIR}/detectors/{det}/checkpoint/pretrained/weights/best.pt'
    dst_dir = f'{REPO_ROOT}/detectors/{det}/checkpoint/pretrained/weights'
    os.makedirs(dst_dir, exist_ok=True)
    if os.path.exists(src):
        shutil.copy2(src, f'{dst_dir}/best.pt')
        print(f'{det}: OK')
    else:
        print(f'{det}: NOT FOUND — check path')

CLIP-D: OK
NPR: OK
R50_nodown: OK


In [6]:
OLD = "'realFORLAB':['FORLAB']"
NEW = ("'realFORLAB':['FORLAB'],\n"
       "        'sd35':['StableDiffusion3.5'],\n"
       "        'sdxl':['StableDiffusionXL'],\n"
       "        'flux':['FLUX.1']")

files_to_patch = [
    'detectors/CLIP-D/utils/dataset.py',
    'detectors/R50_nodown/utils/dataset.py',
    'detectors/NPR/data/__init__.py',
]
for path in files_to_patch:
    with open(path) as f:
        content = f.read()
    if 'sd35' in content:
        print(f'Already patched: {path}')
        continue
    if OLD not in content:
        print(f'ERROR: pattern not found in {path}')
        continue
    with open(path, 'w') as f:
        f.write(content.replace(OLD, NEW))
    with open(path) as f:
        check = f.read()
    print(f'{"OK" if "sd35" in check else "FAILED"}: {path}')

OK: detectors/CLIP-D/utils/dataset.py
OK: detectors/R50_nodown/utils/dataset.py
OK: detectors/NPR/data/__init__.py


In [7]:
import os, shutil, glob

DRIVE     = '/content/drive/MyDrive'
REPO_ROOT = '/content/Image-Deepfake-Detectors-Public-Library'
DEMO      = os.path.join(REPO_ROOT, 'demo_images', 'PreSocial')

GEN_MAP = {
    'sd15':  'StableDiffusion1.5',
    'sd3':   'StableDiffusion3',
    'sd35':  'StableDiffusion3.5',
    'sdxl':  'StableDiffusionXL',
    'flux':  'FLUX.1',
}

demo_root = os.path.join(REPO_ROOT, 'demo_images')
if os.path.exists(demo_root):
    shutil.rmtree(demo_root)

real_dst = os.path.join(DEMO, 'Real', 'FORLAB', 'images')
os.makedirs(real_dst, exist_ok=True)

real_src_files = (
    glob.glob(f'{DRIVE}/FORLAB/*.jpg') +
    glob.glob(f'{DRIVE}/FORLAB/*.png')
)
for src in sorted(real_src_files):
    shutil.copy2(src, os.path.join(real_dst, os.path.basename(src)))
print(f'Real images: {len(real_src_files)} copied')


img2img_folders = {
    'img2img_medieval':     f'{DRIVE}/img2img_images/img2img_medieval',
    'img2img_origami':      f'{DRIVE}/img2img_images/img2img_origami',
    'img2img_vangogh':      f'{DRIVE}/img2img_images/img2img_van gogh style',
    'img2img_pastel':       f'{DRIVE}/img2img_images/img_2img_pastel',
}

img2img_count = {}
for prompt_key, folder in img2img_folders.items():
    if not os.path.exists(folder):
        print(f'MISSING: {folder}')
        continue
    files = (
        glob.glob(f'{folder}/*.png') +
        glob.glob(f'{folder}/*.jpg')
    )
    for fpath in sorted(files):
        fname = os.path.basename(fpath)
        parts = os.path.splitext(fname)[0].split('_')
        if len(parts) < 3:
            continue
        model_key = parts[1]
        quant     = parts[2]
        if model_key not in GEN_MAP:
            continue
        gen_folder = GEN_MAP[model_key]
        dst_dir = os.path.join(DEMO, 'Fake', gen_folder, 'images')
        os.makedirs(dst_dir, exist_ok=True)
        new_name = f'img2img_{prompt_key}_{fname}'
        shutil.copy2(fpath, os.path.join(dst_dir, new_name))
        key = f'{gen_folder}/{quant}'
        img2img_count[key] = img2img_count.get(key, 0) + 1

print('\nImg2img fake images copied:')
for k, v in sorted(img2img_count.items()):
    print(f'  {k}: {v}')

txt2img_root = f'{DRIVE}/txt2img_images'
txt2img_count = {}

for folder in sorted(os.listdir(txt2img_root)):
    folder_path = os.path.join(txt2img_root, folder)
    if not os.path.isdir(folder_path):
        continue
    parts = folder.split('_', 1)
    if len(parts) < 2:
        continue
    model_key = parts[0]
    quant     = parts[1]
    if model_key not in GEN_MAP:
        continue
    gen_folder = GEN_MAP[model_key]
    dst_dir = os.path.join(DEMO, 'Fake', gen_folder, 'images')
    os.makedirs(dst_dir, exist_ok=True)
    files = (
        glob.glob(f'{folder_path}/*.png') +
        glob.glob(f'{folder_path}/*.jpg')
    )
    for fpath in sorted(files):
        fname    = os.path.basename(fpath)
        new_name = f'txt2img_{fname}'
        shutil.copy2(fpath, os.path.join(dst_dir, new_name))
    key = f'{gen_folder}/{quant}'
    txt2img_count[key] = txt2img_count.get(key, 0) + len(files)

print('\nTxt2img fake images copied:')
for k, v in sorted(txt2img_count.items()):
    print(f'  {k}: {v}')

total_fake = sum(img2img_count.values()) + sum(txt2img_count.values())
print(f'\nTotal real: {len(real_src_files)}')
print(f'Total fake: {total_fake}')
print(f'Grand total: {len(real_src_files) + total_fake}')

Real images: 1702 copied

Img2img fake images copied:
  StableDiffusion1.5/fp16: 120
  StableDiffusion1.5/fp4: 120
  StableDiffusion1.5/fp8: 120
  StableDiffusion3.5/fp16: 120
  StableDiffusion3.5/fp4: 119
  StableDiffusion3.5/fp8: 120
  StableDiffusion3/fp16: 120
  StableDiffusion3/fp4: 120
  StableDiffusion3/fp8: 120

Txt2img fake images copied:
  FLUX.1/fp16: 95
  FLUX.1/fp4: 150
  FLUX.1/fp8: 90
  StableDiffusion1.5/fp16: 200
  StableDiffusion1.5/fp4: 198
  StableDiffusion1.5/fp8: 200
  StableDiffusion3.5/fp16: 200
  StableDiffusion3.5/fp4: 200
  StableDiffusion3.5/fp8: 200
  StableDiffusion3/fp16: 199
  StableDiffusion3/fp4: 200
  StableDiffusion3/fp8: 200
  StableDiffusionXL/fp16: 200
  StableDiffusionXL/fp4: 200
  StableDiffusionXL/fp8: 162

Total real: 1702
Total fake: 3773
Grand total: 5475


In [8]:
import os, glob

REPO_ROOT = '/content/Image-Deepfake-Detectors-Public-Library'
DEMO      = os.path.join(REPO_ROOT, 'demo_images', 'PreSocial')

print('=== REAL ===')
real_imgs = glob.glob(f'{DEMO}/Real/FORLAB/images/*')
print(f'FORLAB: {len(real_imgs)} images')

print('\n=== FAKE by generator ===')
for gen in sorted(os.listdir(f'{DEMO}/Fake')):
    imgs    = glob.glob(f'{DEMO}/Fake/{gen}/images/*')
    i2i     = [f for f in imgs if 'img2img' in os.path.basename(f)]
    t2i     = [f for f in imgs if 'txt2img' in os.path.basename(f)]
    print(f'  {gen}: {len(imgs)} total  ({len(i2i)} img2img + {len(t2i)} txt2img)')

print('\n=== FAKE by quantization level (PRIMARY AXIS) ===')
all_fakes = glob.glob(f'{DEMO}/Fake/*/images/*')
for q in ['fp4', 'fp8', 'fp16']:
    q_imgs = [f for f in all_fakes
              if f'_{q}_' in os.path.basename(f) or f'_{q}.' in os.path.basename(f)]
    i2i = [f for f in q_imgs if 'img2img' in os.path.basename(f)]
    t2i = [f for f in q_imgs if 'txt2img' in os.path.basename(f)]
    print(f'  {q}: {len(q_imgs)} total  ({len(i2i)} img2img + {len(t2i)} txt2img)')

=== REAL ===
FORLAB: 1702 images

=== FAKE by generator ===
  FLUX.1: 335 total  (0 img2img + 335 txt2img)
  StableDiffusion1.5: 958 total  (360 img2img + 598 txt2img)
  StableDiffusion3: 959 total  (360 img2img + 599 txt2img)
  StableDiffusion3.5: 959 total  (359 img2img + 600 txt2img)
  StableDiffusionXL: 562 total  (0 img2img + 562 txt2img)

=== FAKE by quantization level (PRIMARY AXIS) ===
  fp4: 1307 total  (359 img2img + 948 txt2img)
  fp8: 1212 total  (360 img2img + 852 txt2img)
  fp16: 1254 total  (360 img2img + 894 txt2img)


In [ ]:
import os, json, sys, argparse

REPO_ROOT = '/content/Image-Deepfake-Detectors-Public-Library'
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

for key in list(sys.modules.keys()):
    if any(x in key for x in ['launcher','dataset','datasets','data']):
        del sys.modules[key]

import launcher

def build_split_json(root_path, out_path):
    test_entries = []
    for mod in ['PreSocial', 'Facebook', 'Telegram', 'X']:
        mod_path = os.path.join(root_path, mod)
        if not os.path.isdir(mod_path):
            continue
        for dirpath, dirnames, filenames in os.walk(mod_path, followlinks=True):
            imgs = [f for f in filenames
                    if os.path.splitext(f)[1].lower() in ['.png','.jpg','.jpeg']]
            if not imgs:
                continue
            rel   = os.path.relpath(dirpath, mod_path)
            parts = rel.split(os.sep)
            if len(parts) < 3:
                continue
            gen = parts[1]
            sub = parts[2]
            for fname in sorted(imgs):
                stem = os.path.splitext(fname)[0]
                test_entries.append(os.path.join(gen, sub, stem))
    unique = sorted(set(test_entries))
    with open(out_path, 'w') as f:
        json.dump({'test': unique}, f, indent=2)
    print(f'Split file: {len(unique)} entries')

DATA_KEYS = 'realFORLAB:pre&sd15:pre&sd3:pre&sd35:pre&sdXL:pre&flux:pre'

orig_load_config = launcher.load_config
def patched_load_config(config_path):
    config = orig_load_config(config_path)
    config['testing'] = [DATA_KEYS]
    return config

launcher.run_demo.__globals__['build_demo_split_json'] = build_split_json
launcher.run_demo.__globals__['load_config'] = patched_load_config

DEMO_ROOT  = os.path.join(REPO_ROOT, 'demo_images')
split_path = os.path.join(REPO_ROOT, 'split_demo.json')
build_split_json(DEMO_ROOT, split_path)

with open(split_path) as f:
    data = json.load(f)
print(f'Total entries: {len(data["test"])}')
print('Sample:', data['test'][:3])

import torch
print(f'\nCUDA: {torch.cuda.is_available()}')

for detector in ['CLIP-D', 'NPR', 'R50_nodown']:
    print(f'\n{"+"*55}')
    print(f'Running: {detector}')
    print(f'{"+"*55}')
    launcher.run_demo(argparse.Namespace(
        demo=True, demo_detector=detector, config_dir='configs',
        weights_name=None, detect=False, image=None,
        weights='pretrained', output=None, dry_run=False,
        detector=None, phases='both',
    ))
    print(f'{detector} done.')

Split file: 5475 entries
Total entries: 5475
Sample: ['FLUX.1/images/txt2img_flux_fp16_p0_seed123', 'FLUX.1/images/txt2img_flux_fp16_p0_seed124', 'FLUX.1/images/txt2img_flux_fp16_p0_seed125']

CUDA: True

+++++++++++++++++++++++++++++++++++++++++++++++++++++++
Running: CLIP-D
+++++++++++++++++++++++++++++++++++++++++++++++++++++++
[demo] Running CLIP-D test with args: --name "demo" --task test --device cuda:0 --split_file /content/Image-Deepfake-Detectors-Public-Library/split_demo.json --data_root /content/Image-Deepfake-Detectors-Public-Library/demo_images --data_keys "realFORLAB:pre&sd15:pre&sd3:pre&sd35:pre&sdXL:pre&flux:pre" --num_threads 8 --arch opencliplinearnext_clipL14commonpool --norm_type clip --resize_size 200 --resize_ratio 1 --resize_prob 0.2 --cmp_qual 65,100 --cmp_prob 0.5 --resizeSize 224
[demo] Completed. Results saved under detectors/<method>/results/demo/<scenario>/results.csv
CLIP-D done.

+++++++++++++++++++++++++++++++++++++++++++++++++++++++
Running: NPR
+++++++

In [ ]:
import glob, pandas as pd, os
from google.colab import files

REPO_ROOT = '/content/Image-Deepfake-Detectors-Public-Library'

dfs = []
for csv_path in glob.glob(f'{REPO_ROOT}/detectors/*/results/**/*.csv', recursive=True):
    det = csv_path.replace(REPO_ROOT+'/detectors/','').split('/')[0]
    df  = pd.read_csv(csv_path, header=0, names=['path','score','label'])
    df['path']     = df['path'].str.strip()
    df['detector'] = det
    dfs.append(df)
    print(f'{det}: {len(df)} rows')

if not dfs:
    print('No results found.')
else:
    combined = pd.concat(dfs, ignore_index=True).drop_duplicates(subset=['path','detector'])

    def extract_quant(p):
        for q in ['fp4','fp8','fp16']:
            if f'_{q}_' in p or f'_{q}.' in p: return q
        return 'unknown'

    def extract_mode(p):
        if 'txt2img' in p: return 'txt2img'
        if 'img2img' in p: return 'img2img'
        return 'real'

    def extract_generator(p):
        for g in ['StableDiffusion3.5','StableDiffusion3','StableDiffusion1.5',
                  'StableDiffusionXL','FLUX.1','FORLAB']:
            if g in p: return g
        return 'unknown'

    combined['quant']     = combined['path'].apply(extract_quant)
    combined['mode']      = combined['path'].apply(extract_mode)
    combined['generator'] = combined['path'].apply(extract_generator)

    print(f'\nTotal rows: {len(combined)}')
    print('\nRows per detector/label:')
    print(combined.groupby(['detector','label']).size().to_string())
    print('\nFake rows per quantization level:')
    print(combined[combined['label']==1].groupby(['detector','quant']).size().to_string())

    out = '/content/all_results.csv'
    combined.to_csv(out, index=False)
    files.download(out)
    print(f'\nSaved: {out}')

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score, accuracy_score

df = pd.read_csv('/content/all_results.csv')
df.columns = df.columns.str.strip()
df['path'] = df['path'].str.strip()

def get_probs(s): return 1/(1+np.exp(-np.array(s, dtype=float)))
def get_preds(det, s):
    p = get_probs(s)
    return (p>0.5).astype(int) if det=='NPR' else (np.array(s)>0).astype(int)

print('='*60)
print('1. OVERALL BASELINE (default threshold)')
print('='*60)
for det, g in df.groupby('detector'):
    s=g['score'].values; l=g['label'].values
    pr=get_probs(s); pd_=get_preds(det,s)
    f=l==1; r=l==0
    tpr=accuracy_score(l[f],pd_[f]) if f.sum() else 0
    tnr=accuracy_score(l[r],pd_[r]) if r.sum() else 0
    auc=roc_auc_score(l,pr) if len(np.unique(l))>1 else 0
    print(f'  {det:12s} TPR={tpr:.3f} TNR={tnr:.3f} AUC={auc:.3f}  real={r.sum()} fake={f.sum()}')

fake = df[df['label']==1.0]

print('\n'+'='*60)
print('2. TPR PER QUANTIZATION LEVEL — PRIMARY RESEARCH QUESTION')
print('='*60)
for det, dg in fake.groupby('detector'):
    print(f'\n  {det}:')
    for q in ['fp4','fp8','fp16']:
        qg = dg[dg['quant']==q]
        if not len(qg): continue
        preds = get_preds(det, qg['score'].values)
        print(f'    {q}: TPR={preds.mean():.3f}  n={len(qg)}')

print('\n'+'='*60)
print('3. TPR PER QUANTIZATION LEVEL PER GENERATOR')
print('='*60)
for det, dg in fake.groupby('detector'):
    print(f'\n  {det}:')
    for gen, gg in dg.groupby('generator'):
        print(f'    {gen}:')
        for q in ['fp4','fp8','fp16']:
            qg = gg[gg['quant']==q]
            if not len(qg): continue
            preds = get_preds(det, qg['score'].values)
            print(f'      {q}: TPR={preds.mean():.3f}  n={len(qg)}')

print('\n'+'='*60)
print('4. TPR PER QUANTIZATION LEVEL PER MODE (img2img vs txt2img)')
print('='*60)
for det, dg in fake.groupby('detector'):
    print(f'\n  {det}:')
    for mode in ['img2img','txt2img']:
        mg = dg[dg['mode']==mode]
        if not len(mg): continue
        print(f'    {mode}:')
        for q in ['fp4','fp8','fp16']:
            qg = mg[mg['quant']==q]
            if not len(qg): continue
            preds = get_preds(det, qg['score'].values)
            print(f'      {q}: TPR={preds.mean():.3f}  n={len(qg)}')

print('\n'+'='*60)
print('5. SCORE DISTRIBUTIONS (threshold miscalibration diagnostic)')
print('='*60)
for det, g in df.groupby('detector'):
    r = g[g['label']==0.0]['score']
    f = g[g['label']==1.0]['score']
    print(f'\n  {det}:')
    print(f'    Real: mean={r.mean():.3f} std={r.std():.3f} min={r.min():.3f} max={r.max():.3f}')
    print(f'    Fake: mean={f.mean():.3f} std={f.std():.3f} min={f.min():.3f} max={f.max():.3f}')

In [ ]:
# CELL 11: Improvements A-E
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score
from sklearn.model_selection import train_test_split
from google.colab import files

df = pd.read_csv('/content/all_results.csv')
df.columns = df.columns.str.strip()
df['path'] = df['path'].str.strip()

def get_probs(s): return 1/(1+np.exp(-np.array(s, dtype=float)))

def find_best_threshold(labels, probs):
    best_t, best_ba = 0.5, 0
    for t in np.percentile(probs, np.arange(1,100)):
        preds=(probs>t).astype(int)
        ba=balanced_accuracy_score(labels,preds)
        if ba>best_ba: best_ba,best_t=ba,t
    return best_t, best_ba

def evaluate(labels, preds, probs):
    f=labels==1; r=labels==0
    tpr=accuracy_score(labels[f],preds[f]) if f.sum() else 0
    tnr=accuracy_score(labels[r],preds[r]) if r.sum() else 0
    ba=balanced_accuracy_score(labels,preds)
    auc=roc_auc_score(labels,probs) if len(np.unique(labels))>1 else 0
    return tpr,tnr,ba,auc

available = [d for d in ['CLIP-D','NPR','R50_nodown'] if d in df['detector'].unique()]

print('='*60)
print('IMPROVEMENT A: THRESHOLD CALIBRATION')
print('(cal/test split stratified by label + quantization)')
print('='*60)

cal_splits={}; test_splits={}; cal_thresholds={}

for det, g in df.groupby('detector'):
    if len(g['label'].unique())<2: continue
    g = g.copy()
    # Stratify by both label and quant to ensure balance across quantization levels
    g['strat_key'] = g['label'].astype(str)+'_'+g['quant'].fillna('unknown')
    min_count = g['strat_key'].value_counts().min()
    if min_count >= 2:
        cal, test = train_test_split(g, test_size=0.5, stratify=g['strat_key'], random_state=42)
    else:
        cal, test = train_test_split(g, test_size=0.5, stratify=g['label'], random_state=42)
    cal_splits[det]=cal; test_splits[det]=test
    best_t,_=find_best_threshold(cal['label'].values, get_probs(cal['score'].values))
    cal_thresholds[det]=best_t
    tp=get_probs(test['score'].values)
    tpr,tnr,ba,auc=evaluate(test['label'].values,(tp>best_t).astype(int),tp)
    print(f'  {det:12s} threshold={best_t:.4f} TPR={tpr:.3f} TNR={tnr:.3f} BalAcc={ba:.3f} AUC={auc:.3f}')

print('\n  Calibrated TPR per quantization level (test split):')
for det in available:
    t=cal_thresholds.get(det,0.5)
    test=test_splits[det]
    fake_test=test[test['label']==1.0]
    print(f'  {det}:')
    for q in ['fp4','fp8','fp16']:
        qg=fake_test[fake_test['quant']==q]
        if not len(qg): continue
        preds=(get_probs(qg['score'].values)>t).astype(int)
        print(f'    {q}: TPR={preds.mean():.3f}  n={len(qg)}')

print('\n'+'='*60)
print('IMPROVEMENT B+C: NORMALIZED WEIGHTED ENSEMBLE')
print('(weights from cal split only, evaluated on test split)')
print('='*60)

def build_pivot(data):
    p=data.pivot_table(index='path',columns='detector',values='score',aggfunc='first')
    meta=data.drop_duplicates('path').set_index('path')
    for col in ['label','quant','generator','mode']:
        if col in meta.columns: p[col]=meta[col]
    return p.dropna(subset=available+['label'])

cal_pivot  = build_pivot(pd.concat([cal_splits[d] for d in available if d in cal_splits]))
test_pivot = build_pivot(pd.concat([test_splits[d] for d in available if d in test_splits]))

auc_weights={}
for det in available:
    if det not in cal_pivot.columns: continue
    mu=cal_pivot[det].mean(); sigma=cal_pivot[det].std()+1e-8
    cal_pivot[f'norm_{det}']  = (cal_pivot[det]-mu)/sigma
    test_pivot[f'norm_{det}'] = (test_pivot[det]-mu)/sigma
    cal_pivot[f'prob_{det}']  = get_probs(cal_pivot[f'norm_{det}'].values)
    test_pivot[f'prob_{det}'] = get_probs(test_pivot[f'norm_{det}'].values)
    auc_weights[det]=roc_auc_score(cal_pivot['label'].values,cal_pivot[f'prob_{det}'].values)

total_auc=sum(auc_weights.values())
print(f'  AUC weights (cal split): { {k:round(v/total_auc,3) for k,v in auc_weights.items()} }')

for piv in [cal_pivot,test_pivot]:
    piv['ensemble_simple']   = piv[[f'prob_{d}' for d in auc_weights]].mean(axis=1)
    piv['ensemble_weighted'] = sum((auc_weights[d]/total_auc)*piv[f'prob_{d}'] for d in auc_weights)

ens_thresholds={}
for name,col in [('Simple','ensemble_simple'),('AUC-weighted','ensemble_weighted')]:
    best_t,_=find_best_threshold(cal_pivot['label'].values,cal_pivot[col].values)
    ens_thresholds[col]=best_t
    tp=test_pivot[col].values
    tpr,tnr,ba,auc=evaluate(test_pivot['label'].values,(tp>best_t).astype(int),tp)
    print(f'  {name:20s} threshold={best_t:.4f} TPR={tpr:.3f} TNR={tnr:.3f} BalAcc={ba:.3f} AUC={auc:.3f}')

best_col='ensemble_weighted'; best_t=ens_thresholds[best_col]

# Only report ensemble as improvement if it outperforms best single detector
best_single_ba = max(
    balanced_accuracy_score(
        test_pivot['label'].values,
        (test_pivot[f'prob_{d}'].values > cal_thresholds[d]).astype(int)
    ) for d in auc_weights
)
ens_ba = balanced_accuracy_score(
    test_pivot['label'].values,
    (test_pivot[best_col].values > best_t).astype(int)
)
if ens_ba >= best_single_ba:
    print(f'\n  ✓ Ensemble improves over best single detector (BalAcc {best_single_ba:.3f} -> {ens_ba:.3f})')
else:
    print(f'\n  ✗ Ensemble ({ens_ba:.3f}) does NOT outperform best single detector ({best_single_ba:.3f})')
    print('    Explanation: averaging with weaker detectors (NPR) dilutes the strong CLIP-D signal.')
    print('    Ensemble is included for completeness but not claimed as an improvement.')

print('\n  Ensemble TPR per quantization level (test split):')
for q in ['fp4','fp8','fp16']:
    m=(test_pivot['quant']==q)&(test_pivot['label']==1)
    if not m.sum(): continue
    p=(test_pivot.loc[m,best_col].values>best_t).astype(int)
    print(f'    {q}: TPR={p.mean():.3f}  n={m.sum()}')

print('\n  Ensemble TPR per generator (test split):')
for gen in sorted(test_pivot['generator'].unique()):
    m=(test_pivot['generator']==gen)&(test_pivot['label']==1)
    if not m.sum(): continue
    p=(test_pivot.loc[m,best_col].values>best_t).astype(int)
    print(f'    {gen:25s}: TPR={p.mean():.3f}  n={m.sum()}')

if 'mode' in test_pivot.columns:
    print('\n  Ensemble TPR per mode (test split):')
    for mode in ['img2img','txt2img']:
        m=(test_pivot['mode']==mode)&(test_pivot['label']==1)
        if not m.sum(): continue
        p=(test_pivot.loc[m,best_col].values>best_t).astype(int)
        print(f'    {mode}: TPR={p.mean():.3f}  n={m.sum()}')

print('\n'+'='*60)
print('IMPROVEMENT D: BOOTSTRAP CONFIDENCE INTERVALS (95%, test split)')
print('='*60)

N_BOOT=1000; rng=np.random.default_rng(42)

def bootstrap_metrics(labels, probs, threshold):
    aucs,tprs,bas=[],[],[]
    n=len(labels)
    for _ in range(N_BOOT):
        idx=rng.integers(0,n,n); l,p=labels[idx],probs[idx]
        if len(np.unique(l))<2: continue
        preds=(p>threshold).astype(int)
        aucs.append(roc_auc_score(l,p))
        f_=l==1
        tprs.append(accuracy_score(l[f_],preds[f_]) if f_.sum() else 0)
        bas.append(balanced_accuracy_score(l,preds))
    return np.array(aucs),np.array(tprs),np.array(bas)

test_labels=test_pivot['label'].values
cal_thresholds_norm={}
for det in auc_weights:
    t,_=find_best_threshold(cal_pivot['label'].values,cal_pivot[f'prob_{det}'].values)
    cal_thresholds_norm[det]=t

for det in auc_weights:
    aucs,tprs,bas=bootstrap_metrics(
        test_labels, test_pivot[f'prob_{det}'].values, cal_thresholds_norm[det])
    print(f'  {det:12s}')
    print(f'    AUC:    {np.mean(aucs):.3f} [{np.percentile(aucs,2.5):.3f}, {np.percentile(aucs,97.5):.3f}]')
    print(f'    TPR:    {np.mean(tprs):.3f} [{np.percentile(tprs,2.5):.3f}, {np.percentile(tprs,97.5):.3f}]')
    print(f'    BalAcc: {np.mean(bas):.3f} [{np.percentile(bas,2.5):.3f}, {np.percentile(bas,97.5):.3f}]')

aucs,tprs,bas=bootstrap_metrics(test_labels,test_pivot[best_col].values,best_t)
print(f'  Ensemble (weighted):')
print(f'    AUC:    {np.mean(aucs):.3f} [{np.percentile(aucs,2.5):.3f}, {np.percentile(aucs,97.5):.3f}]')
print(f'    TPR:    {np.mean(tprs):.3f} [{np.percentile(tprs,2.5):.3f}, {np.percentile(tprs,97.5):.3f}]')
print(f'    BalAcc: {np.mean(bas):.3f} [{np.percentile(bas,2.5):.3f}, {np.percentile(bas,97.5):.3f}]')

print('\n'+'='*60)
print('IMPROVEMENT E: THRESHOLD TRANSFER ACROSS QUANTIZATION LEVELS')
print('Calibrate threshold on one quant level, evaluate on others')
print('This tests whether a calibrated detector generalizes across quantization')
print('='*60)

real_df=df[df['label']==0.0]
quant_levels=['fp4','fp8','fp16']

for det in available:
    print(f'\n  {det}:')
    for cal_q in quant_levels:
        cal_fake=df[(df['detector']==det)&(df['quant']==cal_q)&(df['label']==1)]
        cal_real=real_df[real_df['detector']==det]
        cal_set=pd.concat([cal_fake,cal_real])
        if len(cal_set['label'].unique())<2: continue
        best_t2,_=find_best_threshold(
            cal_set['label'].values, get_probs(cal_set['score'].values))
        row=f'    Cal on {cal_q} -> '
        for test_q in [q for q in quant_levels if q!=cal_q]:
            ts=df[(df['detector']==det)&(df['quant']==test_q)&(df['label']==1)]
            if not len(ts): continue
            tp=(get_probs(ts['score'].values)>best_t2).astype(int)
            row+=f'{test_q}: TPR={tp.mean():.3f} (n={len(ts)})  '
        print(row)

test_pivot.to_csv('/content/ensemble_results.csv')
files.download('/content/ensemble_results.csv')
print('\nAll done.')